# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Arase位置におけるTS04 grad-$B$ driftの推定

Arase L2 orbitとTS04+IGRF磁場モデルから、衛星位置における$B$と$\nabla B$を評価し、LEP-iのエネルギーチャンネルに対するイオンのgrad-$B$ driftを計算する。

pitch angleは$90^\circ$に固定するため、全運動エネルギー$W$は垂直運動エネルギー$W_\perp$に等しい。guiding-center近似におけるdriftは

$$\mathbf{v}_{\nabla B}=\frac{W_\perp}{qB^3}\,\mathbf{B}\times\nabla B.$$

H$^+$、He$^+$、O$^+$はいずれも$q=+e$なので、**同じエネルギーに対するdrift速度は同じ**であり、質量には依存しない。一方、gyroradius $\rho_i=\sqrt{2m_iW_\perp}/(qB)$ は質量に依存する。このnotebookでは$\rho_i/L_B$も計算し、重イオンに対するguiding-center近似の妥当性を確認する。

注意：MGF単点観測から空間勾配は決まらない。ここで得る$\nabla B$はTS04+IGRFモデルの空間勾配であり、観測された局所的な磁場擾乱の勾配ではない。

In [ ]:
import numpy as np

# 設定
TRANGE = ['2022-09-01/22:25:00', '2022-09-01/23:15:00']
CADENCE = '1min'
DIFF_STEP_RE = 0.01       # 中心差分幅 [R_E]
REFERENCE_SPEEDS_KMS = np.array([1.0, 10.0, 100.0])
EFFECT_TIME_S = 60.0      # drift変位を評価する時間幅
EFFECT_FRACTION_LB = 0.1  # EFFECT_TIME_S内に0.1 L_B移動すれば「大きい」とする暫定基準
NO_UPDATE = False         # True: ローカルキャッシュのみ使用

RE_M = 6371.2e3
QE_C = 1.602176634e-19
EV_J = QE_C
AMU_KG = 1.66053906660e-27
SPECIES_MASS_AMU = {'H+': 1.007276, 'He+': 4.002603, 'O+': 15.994915}

assert DIFF_STEP_RE > 0
assert EFFECT_TIME_S > 0


In [ ]:
from pathlib import Path
import os

# SpacePyが初回import時に ~/.spacepy へ書き込めない環境への対策。
# 通常環境では害はなく、設定とキャッシュをworkspace内に限定する。
workspace_root = Path.cwd().resolve().parent if Path.cwd().name == 'KAW_observation' else Path.cwd().resolve()
spacepy_dir = workspace_root / '.spacepy'
spacepy_dir.mkdir(exist_ok=True)
os.environ.setdefault('SPACEPY', str(spacepy_dir))
os.environ.setdefault('MPLCONFIGDIR', str(workspace_root / '.mplconfig'))
Path(os.environ['MPLCONFIGDIR']).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import pyspedas as psp
import pytplot as pt
import geopack
from geopack.models.t04 import t04
from pyspedas.geopack.get_tsy_params import get_tsy_params

print('pyspedas:', getattr(psp, '__version__', 'unknown'))
print('geopack:', getattr(geopack, '__version__', 'unknown'))

## データ取得

- orbit: `erg_orb_l2_pos_gsm`（GSM、想定単位$R_E$）
- LEP-i L2 omniflux: H$^+$=`FPDO`, He$^+$=`FHEDO`, O$^+$=`FODO`
- TS04入力: OMNIの$P_{dyn}$、IMF、速度、密度とKyoto Dstから`get_tsy_params`で作る

LEP-i omnifluxのエネルギー座標はkeV/qである。ここでは全種を一価イオンと仮定するため、数値的にkeVと等しい。

In [ ]:
pt.del_data('*')

psp.projects.erg.orb(
    trange=TRANGE, level='l2', datatype='def',
    time_clip=True, no_update=NO_UPDATE,
)
pos_gsm = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)
if pos_gsm is None:
    raise RuntimeError('erg_orb_l2_pos_gsm was not loaded')
pos_gsm = pos_gsm.sel(time=slice(*TRANGE)).sortby('time')

# 値の大きさとmetadataを検査し、必要な場合だけkm -> R_E変換する。
r_median = float(np.nanmedian(np.linalg.norm(pos_gsm.values, axis=1)))
pos_units_in = str(pos_gsm.attrs.get('units', pos_gsm.attrs.get('Units', 'unknown')))
if r_median > 100.0:
    pos_gsm = pos_gsm / (RE_M / 1e3)
    converted_from_km = True
else:
    converted_from_km = False
pos_gsm.attrs.update(units='R_E', coord_system='GSM')

sample_times = pd.date_range(
    pd.Timestamp(TRANGE[0].replace('/', ' ')),
    pd.Timestamp(TRANGE[1].replace('/', ' ')),
    freq=CADENCE,
)
pos_1min = pos_gsm.interp(time=sample_times)
pos_1min = pos_1min.rename({'v_dim': 'component'}) if 'v_dim' in pos_1min.dims else pos_1min

print('input units attr:', pos_units_in)
print('converted km -> R_E:', converted_from_km)
print('median geocentric distance [R_E]:', float(np.nanmedian(np.linalg.norm(pos_1min.values, axis=1))))
display(pos_1min)

In [ ]:
# LEP-iの実エネルギーチャンネルを取得する（flux値は本計算には使わない）。
psp.projects.erg.lepi(
    trange=TRANGE, level='l2', datatype='omniflux',
    time_clip=True, no_update=NO_UPDATE,
)

LEPI_VARS = {
    'H+': 'erg_lepi_l2_omniflux_FPDO',
    'He+': 'erg_lepi_l2_omniflux_FHEDO',
    'O+': 'erg_lepi_l2_omniflux_FODO',
}

def extract_energy_kevq(tvar):
    data = psp.get_data(tvar)
    if data is None or len(data) < 3:
        raise RuntimeError(f'{tvar}: energy coordinate was not loaded')
    energy = np.asarray(data[2], dtype=float).squeeze()
    if energy.ndim != 1:
        raise ValueError(f'{tvar}: expected 1-D energy coordinate, got {energy.shape}')
    return energy[np.isfinite(energy) & (energy > 0)]

energy_kev = {sp: extract_energy_kevq(vn) for sp, vn in LEPI_VARS.items()}
for sp, en in energy_kev.items():
    print(f'{sp:3s}: {len(en)} channels, {en.min():.4g}--{en.max():.4g} keV/q')
    print(en)

In [ ]:
# TS04 parmod = [Pdyn, Dst, ByIMF, BzIMF, W1, ..., W6]
day0 = pd.Timestamp(TRANGE[0].replace('/', ' ')).floor('D')
day1 = pd.Timestamp(TRANGE[1].replace('/', ' ')).ceil('D')
trange_param = [day0.strftime('%Y-%m-%d/%H:%M:%S'), day1.strftime('%Y-%m-%d/%H:%M:%S')]

psp.projects.kyoto.dst(trange=trange_param)
psp.projects.omni.data(trange=trange_param)
psp.join_vec(['BX_GSE', 'BY_GSM', 'BZ_GSM'])

params_name = get_tsy_params(
    dst_tvar='kyoto_dst',
    imf_tvar='BX_GSE-BY_GSM-BZ_GSM_joined',
    Np_tvar='proton_density',
    Vp_tvar='flow_speed',
    model='ts04',
    pressure_tvar='Pressure',
    speed=True,
    newname='ts04_par',
)
ts04_par = psp.get_data(params_name, xarray=True)
if ts04_par is None:
    raise RuntimeError('TS04 parameters were not created')
ts04_par_1min = ts04_par.interp(time=sample_times, method='nearest')
ts04_columns = ['Pdyn', 'Dst', 'ByIMF', 'BzIMF', 'W1', 'W2', 'W3', 'W4', 'W5', 'W6']

bad_par = ~np.all(np.isfinite(ts04_par_1min.values), axis=1)
print(f'TS04 valid: {(~bad_par).sum()} / {len(bad_par)} samples')
if bad_par.any():
    print('invalid times:', sample_times[bad_par].tolist())
display(ts04_par_1min)

## TS04+IGRF磁場と数値勾配

`t04`が返すのは外部磁場のみなので、内部磁場`igrf_gsm`を加える。各時刻に`geopack.recalc`を実行し、中心点とGSM各軸の$\pm\Delta$で$|\mathbf B|$を評価する。

$$\partial_x B \simeq \frac{B(x+\Delta)-B(x-\Delta)}{2\Delta}.$$

TS04は$X<-15R_E$で有効でない。該当点は明示的にinvalidとする。

In [ ]:
def ts04_total_field_nT(time, position_re, parmod):
    """TS04 external + IGRF internal field in GSM [nT]."""
    position_re = np.asarray(position_re, dtype=float)
    parmod = np.asarray(parmod, dtype=float)
    if position_re.shape != (3,) or parmod.shape != (10,):
        raise ValueError('position must be (3,), parmod must be (10,)')
    if position_re[0] < -15.0:
        raise ValueError('TS04 is invalid tailward of X_GSM = -15 R_E')
    psi = geopack.recalc(pd.Timestamp(time).timestamp())
    bx_ext, by_ext, bz_ext = t04(parmod, psi, *position_re)
    bx_int, by_int, bz_int = geopack.igrf_gsm(*position_re)
    return np.array([bx_ext + bx_int, by_ext + by_int, bz_ext + bz_int], dtype=float)

def field_and_gradB(time, position_re, parmod, step_re=0.01):
    """Return B vector [nT] and grad|B| [nT/R_E], using centered differences."""
    b0 = ts04_total_field_nT(time, position_re, parmod)
    grad = np.empty(3, dtype=float)
    for j in range(3):
        delta = np.zeros(3)
        delta[j] = step_re
        bp = np.linalg.norm(ts04_total_field_nT(time, position_re + delta, parmod))
        bm = np.linalg.norm(ts04_total_field_nT(time, position_re - delta, parmod))
        grad[j] = (bp - bm) / (2.0 * step_re)
    return b0, grad

B_nT = np.full((len(sample_times), 3), np.nan)
gradB_nT_per_RE = np.full_like(B_nT, np.nan)
status = np.full(len(sample_times), 'ok', dtype=object)

for i, time in enumerate(sample_times):
    position = np.asarray(pos_1min.values[i], dtype=float)
    parmod = np.asarray(ts04_par_1min.values[i], dtype=float)
    if not np.all(np.isfinite(position)) or not np.all(np.isfinite(parmod)):
        status[i] = 'missing position/parmod'
        continue
    try:
        B_nT[i], gradB_nT_per_RE[i] = field_and_gradB(time, position, parmod, DIFF_STEP_RE)
    except Exception as exc:
        status[i] = str(exc)

print(pd.Series(status).value_counts())

In [ ]:
# grad-B driftの幾何係数と磁場スケール長
B_T = B_nT * 1e-9
Bmag_T = np.linalg.norm(B_T, axis=1)
gradB_T_per_m = gradB_nT_per_RE * 1e-9 / RE_M

cross_B_gradB = np.cross(B_T, gradB_T_per_m)
# v_gradB = energy_eV * drift_per_eV。q=+eかつ1 eV=e jouleなので係数中でeが相殺する。
drift_vector_per_eV_mps = cross_B_gradB / Bmag_T[:, None]**3
drift_speed_per_eV_mps = np.linalg.norm(drift_vector_per_eV_mps, axis=1)

bhat = B_T / Bmag_T[:, None]
gradB_parallel = np.sum(gradB_T_per_m * bhat, axis=1)[:, None] * bhat
gradB_perp_T_per_m = gradB_T_per_m - gradB_parallel
gradB_perp_mag = np.linalg.norm(gradB_perp_T_per_m, axis=1)
LB_m = Bmag_T / gradB_perp_mag

assert np.allclose(
    drift_speed_per_eV_mps,
    gradB_perp_mag / Bmag_T**2,
    rtol=1e-10, atol=0, equal_nan=True,
)

print('B range [nT]:', np.nanmin(Bmag_T)*1e9, np.nanmax(Bmag_T)*1e9)
print('L_B range [km]:', np.nanmin(LB_m)/1e3, np.nanmax(LB_m)/1e3)

In [ ]:
# species固有のLEP-i energy grid上へ展開する。
vgrad_kms = {}
rho_over_LB = {}
displacement_over_LB = {}

for sp, en_kev in energy_kev.items():
    energy_eV = en_kev * 1e3
    vgrad_kms[sp] = drift_speed_per_eV_mps[:, None] * energy_eV[None, :] / 1e3
    mass_kg = SPECIES_MASS_AMU[sp] * AMU_KG
    rho_m = np.sqrt(2.0 * mass_kg * energy_eV[None, :] * EV_J) / (QE_C * Bmag_T[:, None])
    rho_over_LB[sp] = rho_m / LB_m[:, None]
    displacement_over_LB[sp] = vgrad_kms[sp] * 1e3 * EFFECT_TIME_S / LB_m[:, None]

# 同一energy gridなら全種のdrift速度が同じことを明示的に検証する。
for sp in ('He+', 'O+'):
    if np.array_equal(energy_kev['H+'], energy_kev[sp]):
        assert np.allclose(vgrad_kms['H+'], vgrad_kms[sp], equal_nan=True)


In [ ]:
# 解析結果をxarray Datasetへまとめる。speciesごとにenergy gridが異なり得るため、別変数にする。
result = xr.Dataset(
    data_vars={
        'position_gsm': (('time', 'component'), np.asarray(pos_1min.values)),
        'B_gsm': (('time', 'component'), B_nT),
        'B_magnitude': ('time', Bmag_T * 1e9),
        'grad_B_gsm': (('time', 'component'), gradB_nT_per_RE),
        'L_B': ('time', LB_m / 1e3),
        'status': ('time', status.astype(str)),
    },
    coords={'time': sample_times, 'component': ['x', 'y', 'z']},
    attrs={
        'field_model': 'TS04 external + IGRF internal',
        'pitch_angle_deg': 90.0,
        'finite_difference_step_RE': DIFF_STEP_RE,
        'note': 'grad-B drift is model-derived; not a directly observed spatial gradient',
    },
)

species_tag = {'H+': 'H', 'He+': 'He', 'O+': 'O'}
for sp, tag in species_tag.items():
    edim = f'energy_{tag}'
    result = result.assign_coords({edim: (edim, energy_kev[sp])})
    result[f'v_gradB_{tag}'] = (('time', edim), vgrad_kms[sp])
    result[f'rho_over_LB_{tag}'] = (('time', edim), rho_over_LB[sp])
    result[f'displacement_over_LB_{tag}'] = (('time', edim), displacement_over_LB[sp])
    result[edim].attrs['units'] = 'keV/q (= keV for singly charged ions)'
    result[f'v_gradB_{tag}'].attrs['units'] = 'km/s'

result['position_gsm'].attrs['units'] = 'R_E'
result['B_gsm'].attrs['units'] = 'nT'
result['B_magnitude'].attrs['units'] = 'nT'
result['grad_B_gsm'].attrs['units'] = 'nT/R_E'
result['L_B'].attrs['units'] = 'km'
display(result)

## 「影響が大きい」エネルギーの診断

grad-$B$ drift単独には大小の絶対基準がない。ここでは次の二種類を表にする。

1. $v_{\nabla B}$が指定した基準速度を初めて超えるLEP-i channel
2. 60 sのdrift変位が$0.1L_B$を初めて超えるLEP-i channel

前者の基準速度は研究対象に応じて、観測された$E\times B$ drift、波の垂直位相速度、背景イオン流速などへ置換する必要がある。後者は局所磁場構造を横切る速さの暫定指標であり、普遍的な物理閾値ではない。

In [ ]:
def first_channel_at_or_above(values, energies, threshold):
    ok = np.isfinite(values) & (values >= threshold)
    return float(energies[np.flatnonzero(ok)[0]]) if ok.any() else np.nan

rows = []
for it, time in enumerate(sample_times):
    for sp, energies in energy_kev.items():
        row = {'time': time, 'species': sp}
        for ref in REFERENCE_SPEEDS_KMS:
            row[f'E_at_vgrad>={ref:g}kmps_keV'] = first_channel_at_or_above(
                vgrad_kms[sp][it], energies, ref
            )
        row[f'E_at_dx>={EFFECT_FRACTION_LB:g}LB_keV'] = first_channel_at_or_above(
            displacement_over_LB[sp][it], energies, EFFECT_FRACTION_LB
        )
        row['max_rho_over_LB_in_LEPi'] = np.nanmax(rho_over_LB[sp][it]) if np.any(np.isfinite(rho_over_LB[sp][it])) else np.nan
        rows.append(row)

thresholds = pd.DataFrame(rows).set_index(['time', 'species'])
display(thresholds)
display(thresholds.groupby('species').median(numeric_only=True))

In [ ]:
# 時刻ごとのenergy--drift対応。線の集合は時間変化を表す。
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, constrained_layout=True)
norm = plt.Normalize(mdates.date2num(sample_times[0]), mdates.date2num(sample_times[-1]))
cmap = plt.get_cmap('viridis')

for ax, sp in zip(axes, ('H+', 'He+', 'O+')):
    en = energy_kev[sp]
    for it, time in enumerate(sample_times):
        ax.loglog(en, vgrad_kms[sp][it], color=cmap(norm(mdates.date2num(time))), alpha=0.45, lw=0.9)
    for ref in REFERENCE_SPEEDS_KMS:
        ax.axhline(ref, color='0.4', ls=':', lw=0.8)
    ax.set(title=sp, xlabel='LEP-i energy [keV/q]')
    ax.grid(True, which='both', alpha=0.25)
axes[0].set_ylabel(r'$|v_{\nabla B}|$ [km s$^{-1}$]')
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, ax=axes, pad=0.02)
cbar.set_label('time')
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.show()

In [ ]:
# time--energy mapとguiding-center近似の診断
fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True, constrained_layout=True)
for i, sp in enumerate(('H+', 'He+', 'O+')):
    en = energy_kev[sp]
    pcm0 = axes[i, 0].pcolormesh(sample_times, en, vgrad_kms[sp].T, shading='auto', norm='log', cmap='viridis')
    axes[i, 0].set_yscale('log')
    axes[i, 0].set_ylabel(f'{sp}\nenergy [keV/q]')
    fig.colorbar(pcm0, ax=axes[i, 0], label=r'$|v_{\nabla B}|$ [km/s]')

    pcm1 = axes[i, 1].pcolormesh(sample_times, en, rho_over_LB[sp].T, shading='auto', norm='log', cmap='magma')
    axes[i, 1].set_yscale('log')
    axes[i, 1].contour(sample_times, en, rho_over_LB[sp].T, levels=[0.1, 1.0], colors=['cyan', 'white'], linewidths=1)
    fig.colorbar(pcm1, ax=axes[i, 1], label=r'$\rho_i/L_B$')

axes[0, 0].set_title('TS04 grad-B drift speed')
axes[0, 1].set_title('Guiding-center validity (contours: 0.1, 1)')
for ax in axes[-1]:
    ax.set_xlabel('UTC')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.show()

In [16]:
# 数値微分幅への感度を少数時刻で確認する。
test_indices = np.unique(np.linspace(0, len(sample_times) - 1, min(5, len(sample_times)), dtype=int))
test_steps = np.array([0.005, 0.01, 0.02])
sensitivity_rows = []

for it in test_indices:
    if status[it] != 'ok':
        continue
    for step in test_steps:
        b, grad = field_and_gradB(sample_times[it], pos_1min.values[it], ts04_par_1min.values[it], step)
        bT = b * 1e-9
        gTpm = grad * 1e-9 / RE_M
        coeff = np.linalg.norm(np.cross(bT, gTpm)) / np.linalg.norm(bT)**3
        sensitivity_rows.append({
            'time': sample_times[it], 'step_RE': step,
            'B_nT': np.linalg.norm(b),
            'gradB_nT_per_RE': np.linalg.norm(grad),
            'vgrad_at_10keV_kmps': coeff * 1e4 / 1e3,
        })

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
if not sensitivity.empty:
    display(sensitivity.pivot(index='time', columns='step_RE', values='vgrad_at_10keV_kmps'))

,time,step_RE,B_nT,gradB_nT_per_RE,vgrad_at_10keV_kmps
0,2022-09-01 22:25:00,0.005,325.782381,176.723424,0.701466
1,2022-09-01 22:25:00,0.010,325.782381,176.722982,0.701458
2,2022-09-01 22:25:00,0.020,325.782381,176.721214,0.701423
3,2022-09-01 22:37:00,0.005,300.606666,156.439514,0.739751
4,2022-09-01 22:37:00,0.010,300.606666,156.439107,0.739744
5,2022-09-01 22:37:00,0.020,300.606666,156.437480,0.739714
6,2022-09-01 22:50:00,0.005,277.228583,140.019805,0.765022
7,2022-09-01 22:50:00,0.010,277.228583,140.019462,0.765016
8,2022-09-01 22:50:00,0.020,277.228583,140.018091,0.764993
9,2022-09-01 23:02:00,0.005,260.782291,128.228197,0.783326


step_RE,0.005,0.010,0.020
time,,,
2022-09-01 22:25:00,0.701466,0.701458,0.701423
2022-09-01 22:37:00,0.739751,0.739744,0.739714
2022-09-01 22:50:00,0.765022,0.765016,0.764993
2022-09-01 23:02:00,0.783326,0.783321,0.783301
2022-09-01 23:15:00,0.794271,0.794267,0.794248
